# Fase 1 · Paso 2: descarga automática de los lotes pendientes

Esta libreta reanuda el inventario vigente y procesa secuencialmente todos los lotes pendientes. Cada lote conserva su auditoría y promoción atómica: ante el primer error, la ejecución se detiene sin afectar los lotes ya aprobados. No entrena modelos ni abre la prueba reservada.

## 1. Montar Drive y localizar la raíz privada

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
data_root = Path(private_root)
if not data_root.is_dir():
    raise FileNotFoundError('La raíz privada configurada no está disponible.')
print('Drive montado y raíz privada localizada.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_descarga_automatica')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

Se fija `nda-tools==0.7.0`, versión oficial utilizada en los lotes ya verificados.

In [ ]:
import shutil
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nda-tools==0.7.0', 'keyrings.alt'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Cliente oficial y pruebas automáticas preparados.')

## 4. Recalcular el inventario y determinar el punto de reanudación

La libreta no supone que el siguiente lote sea un número fijo: lo obtiene del inventario privado. Por eso puede volver a ejecutarse después de una desconexión y continuará desde el primer lote realmente pendiente.

In [ ]:
import json

inventory_config = REPO_DIR / 'configs' / 'acquisition_inventory.example.json'
inventory_dir = data_root / 'outputs' / 'auditorias' / 'fase_1' / 'paso_2_inventario'
summary_path = inventory_dir / 'resumen_inventario_publico.json'
queue_path = inventory_dir / 'cola_descarga_selectiva_privada.csv'
frozen_plan = inventory_dir / 'plan_descarga_congelado_privado.csv'
evidence_dir = inventory_dir / 'lotes'
destination_root = data_root / 'dicom' / 'originales' / 'cohorte_v00'

subprocess.run([sys.executable, '-m', 'knee.acquisition_inventory', '--config', str(inventory_config)], cwd=REPO_DIR, check=True)
if not frozen_plan.is_file() or not queue_path.is_file() or not summary_path.is_file():
    raise FileNotFoundError('No se encontraron el inventario y el plan congelado requeridos.')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary['status'] == 'attention_required':
    raise RuntimeError('El inventario requiere revisión; no se iniciará ninguna descarga.')
pending_at_start = list(summary['remaining_batch_numbers'])
print('Punto de reanudación verificado.', {'lotes_pendientes': len(pending_at_start), 'primer_lote_pendiente': summary['first_pending_batch']})

## 5. Autenticar una vez y procesar secuencialmente todos los pendientes

Configura `NDA_USERNAME` y `NDA_PACKAGE_ID` como secretos de Colab. La contraseña se solicita una sola vez, se conserva únicamente en el entorno efímero y se elimina siempre. Cada ciclo prepara, descarga, audita, promueve, actualiza el inventario y registra un lote antes de continuar.

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
import re

nda_username = userdata.get('NDA_USERNAME')
nda_package_id = userdata.get('NDA_PACKAGE_ID')
if not nda_username or not nda_package_id:
    raise RuntimeError('Configura los secretos NDA_USERNAME y NDA_PACKAGE_ID en Colab.')
if not str(nda_package_id).isdigit():
    raise ValueError('NDA_PACKAGE_ID debe ser numérico.')
downloadcmd = shutil.which('downloadcmd')
if not downloadcmd:
    raise RuntimeError('No se encontró el cliente oficial downloadcmd.')

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_root = Path(f'/content/knee_oai_auto_{run_id}')
run_root.mkdir(parents=True)
keyring_root = run_root / '.keyring'
os.environ['XDG_DATA_HOME'] = str(keyring_root)
os.environ['PYTHON_KEYRING_BACKEND'] = 'keyrings.alt.file.PlaintextKeyring'
import keyring

def sanitize_diagnostic(completed, batch_txt, download_dir, nda_log_dir):
    diagnostic = (completed.stdout or '') + '\n' + (completed.stderr or '')
    diagnostic = diagnostic.replace(str(nda_username), '<NDA_USERNAME>')
    diagnostic = diagnostic.replace(str(nda_package_id), '<NDA_PACKAGE_ID>')
    diagnostic = diagnostic.replace(str(batch_txt), '<LISTA_PRIVADA>')
    diagnostic = diagnostic.replace(str(download_dir), '<DESCARGA_TEMPORAL>')
    diagnostic = diagnostic.replace(str(nda_log_dir), '<LOGS_TEMPORALES>')
    diagnostic = re.sub(r's3://\S+', 's3://<RUTA_PRIVADA>', diagnostic)
    return re.sub(r'(?i)\b[0-9]{8}\.tar\.gz(?:\.partial)?\b', '<PAQUETE_PRIVADO>', diagnostic)

completed_batches = []
failed_batch = None
caught_error = None
credential_stored = False
started_at = datetime.now(timezone.utc).isoformat()
nda_password = None
try:
    if pending_at_start:
        nda_password = getpass('Contraseña de la cuenta NDA: ')
        if not nda_password:
            raise ValueError('La contraseña de NDA Tools no puede estar vacía.')
        keyring.set_password('nda-tools', nda_username, nda_password)
        credential_stored = True
        nda_password = None
    while True:
        subprocess.run([sys.executable, '-m', 'knee.acquisition_inventory', '--config', str(inventory_config)], cwd=REPO_DIR, check=True)
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        if summary['status'] == 'attention_required':
            raise RuntimeError('El inventario requiere revisión; la ejecución se detuvo antes del siguiente lote.')
        remaining_batches = list(summary['remaining_batch_numbers'])
        if not remaining_batches:
            break
        batch_number = int(remaining_batches[0])
        failed_batch = batch_number
        lot_root = run_root / f'lote_{batch_number:03d}'
        download_dir = lot_root / 'descarga'
        nda_log_dir = lot_root / 'logs'
        download_dir.mkdir(parents=True)
        nda_log_dir.mkdir(parents=True, exist_ok=True)
        batch_txt = lot_root / f'lote_{batch_number:03d}_s3.txt'
        subprocess.run([sys.executable, '-m', 'knee.selective_download', 'prepare', '--queue', str(queue_path), '--batch', str(batch_number), '--output', str(batch_txt)], cwd=REPO_DIR, check=True)
        print(f'Lote {batch_number}: descarga iniciada; la salida potencialmente sensible permanece oculta.')
        completed = subprocess.run(
            [downloadcmd, '-dp', str(nda_package_id), '-t', str(batch_txt), '-d', str(download_dir), '-wt', '4', '-u', nda_username, '--log-dir', str(nda_log_dir)],
            capture_output=True,
            text=True,
        )
        if completed.returncode != 0:
            diagnostic = sanitize_diagnostic(completed, batch_txt, download_dir, nda_log_dir)
            if 'STATUS 401' in diagnostic or 'Username/password combination is incorrect' in diagnostic:
                raise RuntimeError('NDA rechazó la autenticación. Usa el NDA Username y la contraseña específica de NDA Tools.')
            raise RuntimeError(f'NDA finalizó con código {completed.returncode}. Diagnóstico saneado:\n{diagnostic[-12000:]}')
        subprocess.run([sys.executable, '-m', 'knee.selective_download', 'promote', '--download-dir', str(download_dir), '--batch-file', str(batch_txt), '--destination-root', str(destination_root), '--batch', str(batch_number), '--evidence-dir', str(evidence_dir)], cwd=REPO_DIR, check=True)
        subprocess.run([sys.executable, '-m', 'knee.acquisition_inventory', '--config', str(inventory_config)], cwd=REPO_DIR, check=True)
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        if batch_number in summary['remaining_batch_numbers']:
            raise RuntimeError('El lote promovido sigue figurando como pendiente; se detuvo la secuencia.')
        record = {
            'phase': 1,
            'step': 2,
            'operation': 'selective_download_batch',
            'batch_number': batch_number,
            'executed_at_utc': datetime.now(timezone.utc).isoformat(),
            'git_commit': commit,
            'inventory_status': summary['status'],
            'local_status_counts': summary['local_status_counts'],
            'pending_download_or_replacement': summary['pending_download_or_replacement'],
            'training_executed': False,
            'reserved_test_opened': False,
        }
        evidence_dir.mkdir(parents=True, exist_ok=True)
        (evidence_dir / f'lote_{batch_number:03d}_registro_ejecucion.json').write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
        completed_batches.append(batch_number)
        failed_batch = None
        shutil.rmtree(lot_root)
        print(f'Lote {batch_number}: verificado y cerrado.', {'completados_en_esta_ejecucion': len(completed_batches), 'pendientes': summary['pending_download_or_replacement']})
except Exception as exc:
    caught_error = exc
finally:
    nda_password = None
    try:
        if credential_stored:
            keyring.delete_password('nda-tools', nda_username)
    except Exception:
        pass
    shutil.rmtree(keyring_root, ignore_errors=True)
    final_summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else {}
    run_record = {
        'phase': 1,
        'step': 2,
        'operation': 'automatic_pending_batches',
        'run_id': run_id,
        'started_at_utc': started_at,
        'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': commit,
        'status': 'failed' if caught_error else 'complete',
        'batches_pending_at_start': pending_at_start,
        'batches_completed_in_run': completed_batches,
        'failed_batch': failed_batch,
        'failure_type': type(caught_error).__name__ if caught_error else None,
        'pending_download_or_replacement': final_summary.get('pending_download_or_replacement'),
        'remaining_batch_numbers': final_summary.get('remaining_batch_numbers'),
        'training_executed': False,
        'reserved_test_opened': False,
    }
    evidence_dir.mkdir(parents=True, exist_ok=True)
    (evidence_dir / f'ejecucion_automatica_{run_id}.json').write_text(json.dumps(run_record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
    if caught_error is None:
        shutil.rmtree(run_root, ignore_errors=True)
if caught_error is not None:
    raise caught_error
print('Descarga automática finalizada.', run_record)